# 01 – Data tape profiling & cleaning
**Greek NPL Portfolio Due Diligence & Pricing** · *Synthetic data for practice only – not real bank data.*

Goal: understand the loan tape a seller would send to bidders, find data-quality issues and produce a clean file for valuation.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', '{:,.2f}'.format)

df = pd.read_csv('npl_loan_tape.csv')
print(df.shape)
df.head()

(20085, 18)


,loan_id,borrower_id,segment,region,origination_date,default_date,original_amount,outstanding_principal,accrued_interest,gross_book_value,interest_rate_pct,days_past_due,legal_status,collateral_type,collateral_value_omv,valuation_date,lien_rank,collections_last_12m
0,GR-NPL-007344,B000363,Retail Mortgage,Νότιο Αιγαίο,2005-06-06,2010-04-13,"44,474.43","33,107.07","5,411.72","38,518.79",4.96,6104,Auction,Residential,"26,600.00",2022-10-29,1.00,0.00
1,GR-NPL-003048,B003032,Consumer,Κεντρική Μακεδονία,2007-10-10,2010-06-23,"19,349.03","15,338.76","7,548.52","22,887.28",12.42,6033,Denounced,Unsecured,NaN,NaN,NaN,0.00
2,GR-NPL-018222,B006032,Credit Card,Δυτική Ελλάδα,2009-04-01,2016-10-14,"2,930.23","1,941.88","1,178.36","3,120.24",17.37,3728,Denounced,Unsecured,NaN,NaN,NaN,0.00
3,GR-NPL-012160,B001496,Corporate,Αττική,2009-10-20,2013-09-08,"1,124,525.85","709,745.53","128,396.28","838,141.81",7.29,4860,Denounced,Unsecured,NaN,NaN,NaN,0.00
4,GR-NPL-011297,B011128,Credit Card,Αττική,2009-09-09,2012-08-11,"7,070.08","6,229.58","3,579.94","9,809.52",14.80,5253,Denounced,Unsecured,NaN,NaN,NaN,0.00


## 1. Structure and data types

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20085 entries, 0 to 20084
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   loan_id                20085 non-null  str    
 1   borrower_id            20085 non-null  str    
 2   segment                20085 non-null  str    
 3   region                 20025 non-null  str    
 4   origination_date       20085 non-null  str    
 5   default_date           20085 non-null  str    
 6   original_amount        20085 non-null  float64
 7   outstanding_principal  20085 non-null  float64
 8   accrued_interest       20085 non-null  float64
 9   gross_book_value       20085 non-null  float64
 10  interest_rate_pct      20085 non-null  float64
 11  days_past_due          20085 non-null  int64  
 12  legal_status           20085 non-null  str    
 13  collateral_type        20085 non-null  str    
 14  collateral_value_omv   10110 non-null  float64
 15  valuation_dat

## 2. Data-quality checks

In [3]:
checks = {
    'rows': len(df),
    'exact duplicate rows': df.duplicated().sum(),
    'duplicate loan_id': df['loan_id'].duplicated().sum(),
    'missing region': df['region'].isna().sum(),
    'secured loans without OMV': ((df['collateral_type'] != 'Unsecured') & df['collateral_value_omv'].isna()).sum(),
    'negative OMV': (df['collateral_value_omv'] < 0).sum(),
    'GBV != principal + interest (>1 cent)': ((df['outstanding_principal'] + df['accrued_interest'] - df['gross_book_value']).abs() > 0.01).sum(),
    'default before origination': (pd.to_datetime(df['default_date']) < pd.to_datetime(df['origination_date'])).sum(),
}
pd.Series(checks, name='count').to_frame()

,count
rows,20085
exact duplicate rows,85
duplicate loan_id,85
missing region,60
secured loans without OMV,17
negative OMV,10
GBV != principal + interest (>1 cent),0
default before origination,0


In [4]:
df['segment'].value_counts()

segment
Retail Mortgage    7631
Consumer           5391
SME                3381
Credit Card        3067
Corporate           600
RETAIL MORTGAGE       6
CONSUMER              4
CREDIT CARD           4
CORPORATE             1
Name: count, dtype: int64

In [5]:
print('Total GBV (EUR m):', round(df['gross_book_value'].sum() / 1e6, 1))
df[['original_amount','outstanding_principal','accrued_interest','gross_book_value','days_past_due','collateral_value_omv']].describe().T

Total GBV (EUR m): 2394.1


,count,mean,std,min,25%,50%,75%,max
original_amount,"20,085.00","128,422.46","378,688.75",386.03,"9,333.16","50,982.25","115,944.01","13,891,553.75"
outstanding_principal,"20,085.00","101,030.27","309,671.11",308.77,"7,430.09","37,594.91","87,487.91","13,144,582.41"
accrued_interest,"20,085.00","18,170.06","60,710.01",139.82,"2,358.13","5,600.26","13,028.80","3,348,432.10"
gross_book_value,"20,085.00","119,200.34","368,084.35",460.66,"9,880.22","43,403.45","100,539.76","16,493,014.51"
days_past_due,"20,085.00","4,407.93","1,252.72","1,097.00","3,503.00","4,524.00","5,492.00","6,206.00"
collateral_value_omv,"10,110.00","175,096.04","495,787.99","-414,200.00","43,900.00","77,100.00","140,000.00","18,033,800.00"


## 3. Cleaning
Rules (each one logged):
1. Drop exact duplicate rows (same loan sent twice).
2. Standardise `segment` labels (upper-case variants).
3. Negative OMV is impossible → set to missing.
4. Missing region → `Unknown`.
5. Secured loans without a valid OMV are **kept** but flagged `omv_missing = 1` (valued as unsecured – a conservative choice).

In [6]:
log = []
n0 = len(df)
clean = df.drop_duplicates().copy()
log.append(('Dropped exact duplicate rows', n0 - len(clean)))

seg_map = {s.upper(): s for s in ['Retail Mortgage', 'Consumer', 'Credit Card', 'SME', 'Corporate']}
bad_case = (~clean['segment'].isin(seg_map.values())).sum()
clean['segment'] = clean['segment'].str.upper().map(seg_map)
log.append(('Standardised segment labels', bad_case))

neg = (clean['collateral_value_omv'] < 0).sum()
clean.loc[clean['collateral_value_omv'] < 0, 'collateral_value_omv'] = np.nan
log.append(('Negative OMV set to missing', neg))

miss_reg = clean['region'].isna().sum()
clean['region'] = clean['region'].fillna('Unknown')
log.append(('Missing region set to Unknown', miss_reg))

clean['secured'] = (clean['collateral_type'] != 'Unsecured').astype(int)
clean['omv_missing'] = ((clean['secured'] == 1) & clean['collateral_value_omv'].isna()).astype(int)
log.append(('Secured loans flagged omv_missing', clean['omv_missing'].sum()))

pd.DataFrame(log, columns=['step', 'rows affected'])

,step,rows affected
0,Dropped exact duplicate rows,85
1,Standardised segment labels,15
2,Negative OMV set to missing,10
3,Missing region set to Unknown,60
4,Secured loans flagged omv_missing,23


## 4. Derived fields used in the strats

In [7]:
clean['ltv'] = clean['gross_book_value'] / clean['collateral_value_omv']
clean['ltv_bucket'] = pd.cut(clean['ltv'], [0, 0.8, 1.0, 1.2, 1.5, np.inf],
                             labels=['<80%', '80-100%', '100-120%', '120-150%', '>150%'])
clean['ltv_bucket'] = clean['ltv_bucket'].astype(str).replace('nan', 'Unsecured / no OMV')
clean['dpd_years'] = clean['days_past_due'] / 365
clean['dpd_bucket'] = pd.cut(clean['dpd_years'], [0, 5, 10, 15, np.inf], labels=['3-5y', '5-10y', '10-15y', '15y+'])
clean[['ltv_bucket', 'dpd_bucket']].apply(lambda s: s.value_counts()).fillna(0).astype(int)

,ltv_bucket,dpd_bucket
10-15y,0,9310
100-120%,1607,0
120-150%,1843,0
15y+,0,5074
3-5y,0,554
5-10y,0,5062
80-100%,1802,0
<80%,2404,0
>150%,2404,0


In [8]:
print(clean.shape, '| duplicates left:', clean.duplicated().sum())
print('Clean GBV (EUR m):', round(clean['gross_book_value'].sum() / 1e6, 1))
clean.to_csv('npl_loan_tape_clean.csv', index=False)

(20000, 24) | duplicates left: 0
Clean GBV (EUR m): 2380.8
